In [ ]:
# STEP 1 (run first, once): Clones the ReMasker repo and cd's into it.

!git clone https://github.com/alps-lab/remasker.git
%cd remasker

Cloning into 'remasker'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 48 (delta 7), reused 41 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 1.08 MiB | 2.95 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/remasker/remasker


In [ ]:
# STEP 2 (run once per session): Installs dependencies (timm, hyperimpute).

!pip install timm hyperimpute

In [ ]:
# STEP 3 (run once, after STEP 1): Patches utils.py for new NumPy (np.float -> float).
# The grep below should print no remaining 'np.float)' matches. 

!sed -i 's/dtype=np\.float)/dtype=float)/' utils.py
!grep -n "np\.float" utils.py

In [ ]:
# STEP 4 (run once, after STEP 1): Patches plugin_mae.py, removing the CKA / extra-missing-rate code and the hard-coded '/data/ting/...' path. Prints 'patched' on success.
# DO NOT run this cell twice: the second run raises ValueError.
# If that happens, delete the 'remasker' folder and restart from STEP 1.

src = open("plugin_mae.py").read()

start = src.index("        cka = {} \n")
end = src.index("        self.model.train()\n", start)
src = src[:start] + src[end:]

start = src.index("        miss_rates = [0.1, 0.3, 0.5, 0.7]\n")
end = src.index("        for epoch in range(self.max_epochs):", start)
src = src[:start] + "\n" + src[end:]

start = src.index("            if (epoch + 1) % 10 == 0:\n                # check CKA")
end = src.index("            # if total_loss < best_loss:", start)
src = src[:start] + src[end:]

start = src.index("        with open('/data/ting/remasker2/output/letter-cka'")
end = src.index("        # torch.save(self.model.state_dict(), self.path)\n        return self", start)
src = src[:start] + src[end:]

open("plugin_mae.py", "w").write(src)
print("patched")

patched


In [ ]:
# This is an optional step and can be skipped. It shows the GPU and working directory.
!pwd
!nvidia-smi
!grep -n "ting/remasker2\|letter.pt" plugin_mae.py

/content/remasker/remasker
Tue Oct 20 12:02:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+--------------------

In [ ]:
# Run the baseline imputers on the 'yacht' dataset.
# Each dataset cell after this is independent: run in any order, or only the ones you need.
# Results are saved in directory 'repro'.

!python run_exp.py --dataset yacht --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.2342 +/- 0.0176


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.1094 +/- 0.0135


AUROC score
  Scenario  miss_pct [0, 1] Evaluated: mae
0      MAR              0.3    0.0 +/- 0.0


In [ ]:
%%time
!python run_exp.py --dataset diabetes --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.1393 +/- 0.0118


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.0668 +/- 0.0081


AUROC score
  Scenario  miss_pct [0, 1] Evaluated: mae
0      MAR              0.3    0.0 +/- 0.0
CPU times: user 360 ms, sys: 64.8 ms, total: 425 ms
Wall time: 3min 56s


In [ ]:
%%time
!python run_exp.py --dataset climate --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.2796 +/- 0.0166


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.2592 +/- 0.0275


AUROC score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.9552 +/- 0.0107
CPU times: user 439 ms, sys: 67.8 ms, total: 507 ms
Wall time: 4min 50s


In [ ]:
%%time
!python run_exp.py --dataset credit --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.2427 +/- 0.1019


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.1219 +/- 0.0819


AUROC score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.9265 +/- 0.0015
CPU times: user 809 ms, sys: 153 ms, total: 962 ms
Wall time: 8min 43s


In [ ]:
%%time
!python run_exp.py --dataset raisin --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.0927 +/- 0.0344


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.0438 +/- 0.0185


AUROC score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.9946 +/- 0.0056
CPU times: user 1.27 s, sys: 204 ms, total: 1.47 s
Wall time: 13min 37s


In [ ]:
%%time
!python run_exp.py --dataset compression --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.1156 +/- 0.0123


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.0476 +/- 0.0032


AUROC score
  Scenario  miss_pct [0, 1] Evaluated: mae
0      MAR              0.3    0.0 +/- 0.0
CPU times: user 1.17 s, sys: 173 ms, total: 1.35 s
Wall time: 12min 50s


In [ ]:
%%time
!python run_exp.py --dataset wine --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.1144 +/- 0.0284


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.0345 +/- 0.0041


AUROC score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.7943 +/- 0.0016
CPU times: user 2.32 s, sys: 329 ms, total: 2.65 s
Wall time: 25min 4s


In [22]:
%%time
!python run_exp.py --dataset obesity --exp_name repro --note run1

RMSE score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.1942 +/- 0.0163


Wasserstein score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.0483 +/- 0.0081


AUROC score
  Scenario  miss_pct [0, 1]     Evaluated: mae
0      MAR              0.3  0.9422 +/- 0.0023
CPU times: user 3.09 s, sys: 469 ms, total: 3.56 s
Wall time: 33min 20s
